# prompts_test.ipynb — 提示词与注入链验证 (3用例组)

覆盖 spec 验收表收尾三项: ⑭ 全模板变量集精确匹配且渲染无缺变量 ⑮ Kim Wexler 人设注入反问/finalize 兜底全链路 + LEGAL_ANALYSIS_ROLE 切 Saul 仅影响法律分析(反问人设不变) ⑯ 已知案件要素 digest 注入链(_build_analysis_context 前置 [已知案件要素] 段 + planner/executor 模板注入位)。纯静态校验, 不调 LLM, 仅 import prompts/state/rag_tools。

## Cell 1 环境准备 (setup, 自包含 sys.path — 无 LLM patch, 仅 import prompts)

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "lawApp_LangGraph" else Path.cwd()
sys.path.insert(0, str(ROOT))
from lawApp_LangGraph import prompts
print("setup ok")

## 用例⑭ 全模板渲染无缺变量 (9 个字符串模板变量集不多不少 + 2 个 PromptTemplate 实例渲染)

In [ ]:
from langchain_core.prompts import PromptTemplate

# 字符串模板: 检查变量集合与预期一致(不多不少)
STRING_PROMPTS = {
    "RISK_GATE_PROMPT": {"query"},
    "ELEMENT_ASSESS_PROMPT": {"query", "elements_digest", "last_question",
                              "last_answer", "round", "max_rounds"},
    "MID_CLARIFY_PROMPT": {"query", "top_docs_summary"},
    "REPLAN_CHECK_PROMPT": {"user_query", "executed_summary", "doc_count",
                            "quality_verdict", "web_count", "law_count",
                            "error_info"},
    "PLANNER_SYSTEM": {"available_tools", "query", "elements_digest"},
    "EXECUTOR_PROMPT": {"step_description", "tool_name", "user_query",
                        "elements_digest", "rag_summary", "eval_summary",
                        "law_summary", "web_summary"},
    "REPLANNER_SYSTEM_PROMPT": {"executed_steps", "doc_count", "quality",
                                "web_count", "law_count", "error",
                                "replan_reason", "available_tools",
                                "user_query", "next_id"},
    "DEGRADE_CONFIRM_MSG": {"failed_tool"},
    "BUDGET_CONFIRM_MSG": {"missing"},
}
for name, expected in STRING_PROMPTS.items():
    tpl = getattr(prompts, name)
    assert isinstance(tpl, str), name
    got = set(PromptTemplate.from_template(tpl).input_variables)
    # EXECUTOR_PROMPT 的 {tool_name} 出现两次且 format 需要它 → 在 expected 内即可
    assert got == expected, f"{name}: got {got}, expected {expected}"
    # 渲染不报错
    PromptTemplate.from_template(tpl).format(**{v: "测试值" for v in got})
print("⑭ 字符串模板变量集 OK")

# PromptTemplate 实例
for name, expected in {
    "FINALIZE_CASE_PROMPT": {"docs", "query"},
    "FINALIZE_DIRECT_PROMPT": {"query"},
}.items():
    tpl = getattr(prompts, name)
    assert tpl.format(**{v: "测试值" for v in expected})
print("⑭ 全部模板渲染 OK")

## 用例⑮ Kim Wexler 人设全链路注入 + Saul 切换仅影响法律分析 (env 用后即清)

In [ ]:
import os

assert "Kim Wexler" in prompts.KIM_PERSONA_BLOCK
# Kim 注入反问与兜底
assert "Kim Wexler" in prompts.ELEMENT_ASSESS_PROMPT
assert "Kim Wexler" in prompts.MID_CLARIFY_PROMPT
assert "Kim Wexler" in prompts.FINALIZE_CASE_PROMPT.template
assert "Kim Wexler" in prompts.FINALIZE_DIRECT_PROMPT.template
# 旧傲娇人设已清除
assert "傲娇" not in prompts.FINALIZE_DIRECT_PROMPT.template
assert "傲娇" not in prompts.FINALIZE_CASE_PROMPT.template

os.environ["LEGAL_ANALYSIS_ROLE"] = "kim"
assert "Kim Wexler" in prompts.get_analysis_prompt().template
os.environ["LEGAL_ANALYSIS_ROLE"] = "saul"
assert "Saul Goodman" in prompts.get_analysis_prompt().template
# 反问不随 saul 变
assert "Kim Wexler" in prompts.ELEMENT_ASSESS_PROMPT
os.environ.pop("LEGAL_ANALYSIS_ROLE")
print("⑮ Kim 全链路 + Saul 仅分析 OK")

## 用例⑯ 已知案件要素 digest 注入链 (分析上下文前置段 / 空记录不注入 / 规划与执行模板注入位)

In [ ]:
from lawApp_LangGraph.state import PromptsRecord
from lawApp_LangGraph.tools.rag_tools import _build_analysis_context

pr = PromptsRecord(known_elements="婚姻现状:在婚分居 | 核心诉求:争取抚养权")
ctx = _build_analysis_context(pr)
assert ctx.startswith("[已知案件要素]")
assert "争取抚养权" in ctx

# 空/默认不注入
ctx2 = _build_analysis_context(PromptsRecord())
assert "[已知案件要素]" not in ctx2

# planner/executor 模板含注入位
assert "elements_digest" in prompts.PLANNER_SYSTEM
assert "已知案件要素" in prompts.PLANNER_SYSTEM
assert "已知案件要素" in prompts.EXECUTOR_PROMPT
print("⑯ digest 注入链 OK")
print("ALL PASSED")